In [1]:
#install dependencies
%pip install numpy pandas pydicom nibabel opencv-python-headless tqdm -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, sys, json, hashlib, warnings, logging
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import cv2
import pydicom
import nibabel as nib
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
print('import done')

import done


In [3]:
CONFIG = {
    "ct_path":  '/Users/ayushchhetry/spartis assignment/JUH_MR-CT_dataset/CT/image_CT',
    "mri_path": '/Users/ayushchhetry/spartis assignment/JUH_MR-CT_dataset/MR/image_MR',

    "ct_mask_root":  '/Users/ayushchhetry/spartis assignment/JUH_MR-CT_dataset/CT/mask_CT', #absolute path of human annotated target labels
    "mri_mask_root": '/Users/ayushchhetry/spartis assignment/JUH_MR-CT_dataset/MR/mask_MR',
    
    "output_root": "/Users/ayushchhetry/spartis assignment/dataset", #where all the processed, final 2d slices and labels whill be stored

    #CT windowing in hounsfield unit - represents physical tissue density
    "ct_window_center": 40, #target densities bwteen 0 to 80 hu
    "ct_window_width":  80, #brighter than 80  hu means bones and less than 0 is air that is darker

    #percentile clipping for 1% top/bottom  - normalization method 
    "mri_clip_low":  1.0,
    "mri_clip_high": 99.0,

    "target_size": (256, 256),   #resize
    "min_brain_fraction": 0.02,  #slice quality filter - if a slice contains less than 2% of actual tissue - it is blank

    #data split
    "split_ratios": {"train": 0.70, "val": 0.15, "test": 0.15}, #slices the model will learn from , slices used to evalute model progress, fully isloated slices for final model accuracy
    "seed": 42, #random seed to give accuracy stability
    "save_format": "npy",
}
print('config done')

config done


In [4]:
def setup_logger():
    log = logging.getLogger('pipeline') #logger instance, make sure that logs generated here are isloated from others
    log.setLevel(logging.DEBUG) #debug ensures that logger is capable of capturing everything to the most minor diagnostic detail
    log.handlers.clear()
    fmt = logging.Formatter('%(asctime)s | %(levelname)s | %(message)s', '%H:%M:%S') #level name - debug info warning error critical
    fh = logging.FileHandler('pipeline.log', mode='w'); fh.setFormatter(fmt)
    sh = logging.StreamHandler(sys.stdout);             sh.setFormatter(fmt); sh.setLevel(logging.INFO)
    log.addHandler(fh); log.addHandler(sh)
    return log

LOG = setup_logger()
print('logger done')

#Dual Logging Streams: You keep your Jupyter screen tidy by only printing important milestones (INFO), 
#while secretly writing highly granular diagnostics (DEBUG) to the pipeline.log file behind the scenes.
#Audit Trail: If your pipeline crashes on a dataset containing 10,000 files, you have a physical text file (pipeline.log) that you 
#can read to pinpoint the exact file, time, and reason for the crash.

logger done


In [6]:
#files → keep only valid dicom / nii files → ignore junk → store metadata

def discover_files(root: str, modality: str, is_mask: bool = False) -> List[Dict]: #removes junk,corrupted files and prevent data leakage
    #Find all .dcm / .nii files.
    root = Path(root)
    #exception hnalding
    if not root.exists():
        LOG.error(f'Path not found: {root}'); return []
        
    records = []                             #store info like path,modality - ct or mri ,file type
    for p in sorted(root.rglob('*')):
        if not p.is_file(): continue
        s = p.suffix.lower()                #get only the file extension
        if s not in ('.dcm', '.nii', '.gz'): continue  #skip any non imaging system
        if s == '.gz' and '.nii' not in p.stem: continue
        if p.stat().st_size < 512: #if dicom or nfti file contains data less than 512 byte then remove it
            LOG.warning(f'Skipping tiny file: {p.name}'); continue   #skip files less than 512 byte

        # Safety net: catch mask files that end up under an image root - data leakage
        path_is_mask = 'mask' in str(p).lower()
        if path_is_mask and not is_mask:
            LOG.warning(f'SKIPPED — mask file found in image folder: {p.name}')
            continue
            
        records.append({'path': str(p), 'modality': modality,
                        'file_type': 'dicom' if s == '.dcm' else 'nifti', 'is_mask': is_mask})
    LOG.info(f'{modality} ({"masks" if is_mask else "images"}): found {len(records)} files')
    return records

def validate_and_load(record: Dict) -> Tuple[Optional[np.ndarray], str]: #used for qulaity assurance
    try:
        if record['file_type'] == 'dicom':
            ds   = pydicom.dcmread(record['path'])
            data = ds.pixel_array.astype(np.float32) #extract raw pixle and convert it to float
            if record['modality'] == 'CT':                               #converts raw values → Hounsfield Units (HU)
                data = data * float(getattr(ds, 'RescaleSlope', 1)) + float(getattr(ds, 'RescaleIntercept', 0)) # hu = raw pixel * rescale slope + rescle intercept
                #To get the true Hounsfield Unit values, we must apply the linear equation (Raw×Slope)+Intercept using the rescaling coefficients 
                #stored inside the DICOM header. Without this step, soft tissue windowing or model training would fail because the densities 
                #would be arbitrary.
        else:
            data = nib.load(record['path']).get_fdata().astype(np.float32)

        if data.size == 0:          return None, 'empty' #drop empty array 
        if np.all(data == 0):       return None, 'all_zeros' #drop fully black array
        if np.any(np.isnan(data)):  return None, 'contains_nan' #nan value will corrupt loss calculation
        if np.any(np.isinf(data)):  return None, 'contains_inf' #drop infinite values / very big vlaues
        if data.ndim not in (2,3,4):return None, f'bad_ndim_{data.ndim}' #ensure dimension are 2d,3d,4d
        return data, 'ok'
    except Exception as e:
        return None, f'load_error:{e}'

print('Done')

Done


In [7]:
def preprocess_ct(data: np.ndarray, cfg: dict) -> np.ndarray: #normalization - window clipping
    #window for HU values
    lo = cfg['ct_window_center'] - cfg['ct_window_width'] / 2
    hi = cfg['ct_window_center'] + cfg['ct_window_width'] / 2
    #normalize it to just o and 1s
    return np.clip((np.clip(data, lo, hi) - lo) / (hi - lo + 1e-8), 0, 1).astype(np.float32)

def preprocess_mri(data: np.ndarray, cfg: dict) -> np.ndarray: # normalization - percentile clipping as they dont have standard physical scale
    #percentile clipping
    nz = data[data > 0] #filter out all absolute 0 value
    if nz.size == 0: return data.astype(np.float32)
        #lo is 1% and high is 99%
    lo, hi = np.percentile(nz, cfg['mri_clip_low']), np.percentile(nz, cfg['mri_clip_high'])
    return np.clip((np.clip(data, lo, hi) - lo) / (hi - lo + 1e-8), 0, 1).astype(np.float32)

def extract_and_resize(volume: np.ndarray, cfg: dict) -> List[np.ndarray]: #filter out empty background, resize valid axial slices
    #3d to 2d
    if volume.ndim == 4: volume = volume[..., 0]      # 4d - take first timepoint
    if volume.ndim == 2: volume = volume[:, :, None]  # 2d - treat as 1-slice
    slices = []
    for z in range(volume.shape[2]):
        sl = volume[:, :, z]
        if np.count_nonzero(sl) / sl.size < cfg['min_brain_fraction']: continue
        #resize
        sl = cv2.resize(sl, (cfg['target_size'][1], cfg['target_size'][0]),
                        interpolation=cv2.INTER_LINEAR) #Bilinear interpolation blends neighboring pixels smoothly,
        #store slice
        slices.append(sl.astype(np.float32))
    return slices

def save_slice(sl, out_dir, vol_id, index, fmt): #medical scan - bilinear interpolation and segmentation mask - nearest neibhour interpolation
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    if fmt == 'npy':
        np.save(f'{out_dir}/{vol_id}_z{index:04d}.npy', sl)
    else:
        cv2.imwrite(f'{out_dir}/{vol_id}_z{index:04d}.png',
                    (sl * 255).clip(0,255).astype(np.uint8))
        
def save_mask_slice(sl, out_dir, vol_id, index, fmt):
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    sl_resized = cv2.resize(sl, (CONFIG['target_size'][1], CONFIG['target_size'][0]),
                            interpolation=cv2.INTER_NEAREST)
    if fmt == 'npy':
        np.save(f'{out_dir}/{vol_id}_z{index:04d}.npy', sl_resized.astype(np.uint8)) #0 - background , 1 - liver , 2 - tumor
    else:
        cv2.imwrite(f'{out_dir}/{vol_id}_z{index:04d}.png', sl_resized.astype(np.uint8))

print('preprocessing done')

preprocessing done


In [8]:
def split_records(records: List[Dict], cfg: dict) -> Dict[str, List[Dict]]: #reproducible, ramdomized dataset partitioing
    rng = np.random.default_rng(cfg['seed'])
    index = np.arange(len(records)); rng.shuffle(index)
    n, nt = len(records), int(len(records) * cfg['split_ratios']['train']) #nt= number of records needed for training 
    nv = int(n * cfg['split_ratios']['val']) # number of records required for validation
    return {'train': [records[i] for i in index[:nt]],
            'val':   [records[i] for i in index[nt:nt+nv]],
            'test':  [records[i] for i in index[nt+nv:]]}

print('training spilt done')

training spilt done


In [9]:
def process_volume_list(vols, modality, split, cfg, is_mask):
    subfolder = 'masks' if is_mask else 'images'
    out_dir   = f"{cfg['output_root']}/{split}/{modality}/{subfolder}"
    qc_rows   = []
    manifest  = []   # one row per saved slice
    for rec in tqdm(vols, desc=f'{modality}/{split}/{subfolder}', leave=False):
        row = {**rec, 'split': split, 'slices_saved': 0, 'issue': ''}
        data, status = validate_and_load(rec)
        if status != 'ok':
            row['issue'] = status
            LOG.warning(f'SKIP {Path(rec["path"]).name}: {status}')
        else:
            try:
                vol_id = Path(rec['path']).stem.replace('.nii', '')  #generates a clean patient identifier
                if is_mask:
                    vol = data
                    if vol.ndim == 4: vol = vol[..., 0]
                    if vol.ndim == 2: vol = vol[:, :, None]
                    slices = []
                    for z in range(vol.shape[2]):
                        sl = vol[:, :, z]
                        if np.count_nonzero(sl) / sl.size < cfg['min_brain_fraction']: continue
                        slices.append(sl)
                    for i, sl in enumerate(slices):
                        save_mask_slice(sl, out_dir, vol_id, i, cfg['save_format'])
                else:
                    vol    = preprocess_ct(data, cfg) if modality == 'CT' else preprocess_mri(data, cfg)
                    slices = extract_and_resize(vol, cfg)
                    for i, sl in enumerate(slices):
                        fpath = f'{out_dir}/{vol_id}_z{i:04d}.{cfg["save_format"]}'
                        save_slice(sl, out_dir, vol_id, i, cfg['save_format'])
                        manifest.append({
                            'subject_id':  vol_id,
                            'modality':    modality,
                            'split':       split,
                            'slice_index': i,
                            'image_path':  fpath,
                            'mask_path':   '',   # filled later by build_manifest()
                            'source_file': rec['path'],
                        })
                row['slices_saved'] = len(slices)
            except Exception as e:
                row['issue'] = str(e); LOG.error(e)
        qc_rows.append(row)
    return qc_rows, manifest


def build_manifest(manifest_rows): #build_manifest reads the output directories, maps corresponding image and mask slices together 
    #using a high-speed dictionary lookup, and outputs a unified CSV manifest.
    import pandas as pd
    df = pd.DataFrame(manifest_rows)
    if df.empty:
        return df

    # Separate image and mask entries so we can join them
    img_df  = df[df['modality'].isin(['CT', 'MRI'])].copy()

    # Load mask paths from the output folders and build a lookup
    mask_lookup = {}   # (subject_id, modality, split, slice_index) -> mask_path -- this is used for high speed hash map lookup to pair mask with images
    for modality in ['CT', 'MRI']:
        for split in ['train', 'val', 'test']:
            mask_dir = Path(CONFIG['output_root']) / split / modality / 'masks'
            if not mask_dir.exists():
                continue
            for f in sorted(mask_dir.glob(f'*.{CONFIG["save_format"]}')):
                parts = f.stem.rsplit('_z', 1)
                if len(parts) != 2: continue
                vol_id, z_str = parts[0], parts[1]
                try:
                    z_idx = int(z_str)
                except ValueError:
                    continue
                # FIX: Normalize vol_id by replacing '_m' with '_s' to match the scan names
                normalized_vol_id = vol_id.replace('_m', '_s')
                mask_lookup[(normalized_vol_id, modality, split, z_idx)] = str(f)

    # Fill mask_path column
    def lookup_mask(row):
        key = (row['subject_id'], row['modality'], row['split'], row['slice_index'])
        return mask_lookup.get(key, '')

    img_df['mask_path'] = img_df.apply(lookup_mask, axis=1) #applies lookup mask across every axis =1 containing perfectly paired mask

    # Reorder columns for readability
    cols = ['subject_id', 'modality', 'split', 'slice_index',
            'image_path', 'mask_path', 'source_file']
    img_df = img_df[cols].sort_values(['split', 'modality', 'subject_id', 'slice_index'])
    img_df = img_df.reset_index(drop=True)
    return img_df


def run_pipeline(cfg): #orchestration , makes two file qc report for debugging and data manifest for training
    qc       = []
    manifest = []
    jobs = [
        ('CT',  cfg['ct_path'],       False),
        ('MRI', cfg['mri_path'],      False),
        ('CT',  cfg['ct_mask_root'],  True),
        ('MRI', cfg['mri_mask_root'], True),
    ]
    for modality, root, is_mask in jobs:
        if not root: continue
        records = discover_files(root, modality, is_mask=is_mask)
        splits  = split_records(records, cfg)
        for split, vols in splits.items():
            qc_rows, mf_rows = process_volume_list(vols, modality, split, cfg, is_mask)
            qc.extend(qc_rows)
            manifest.extend(mf_rows)

    # QC report — one row per volume, flags errors
    qc_df = pd.DataFrame(qc)
    qc_df.to_csv('qc_report.csv', index=False)

    # Training manifest — one row per slice, with image + mask path ---- this saves the paired image and mask slice
    manifest_df = build_manifest(manifest)
    manifest_df.to_csv('dataset_manifest.csv', index=False)

    # Summary
    issues = (qc_df['issue'] != '').sum()
    LOG.info(f'Done — {len(manifest_df)} slices in manifest, {issues} volume issues')
    LOG.info(f'Manifest saved → dataset_manifest.csv')
    LOG.info(f'QC report  saved → qc_report.csv')

    print('\n--- Manifest preview (first 5 rows) ---')
    print(manifest_df.head().to_string(index=False))
    print(f'\nTotal slices: {len(manifest_df)}')
    print(manifest_df.groupby(['split', 'modality']).size().rename('slices').to_string())

    return qc_df, manifest_df


QC, MANIFEST = run_pipeline(CONFIG)

print('\nOutput structure:')
for p in sorted(Path(CONFIG['output_root']).rglob('*')):
    if p.is_dir(): print(' ', p)

16:53:23 | INFO | CT (images): found 88 files


CT/train/images:   0%|          | 0/61 [00:00<?, ?it/s]

CT/val/images:   0%|          | 0/13 [00:00<?, ?it/s]

CT/test/images:   0%|          | 0/14 [00:00<?, ?it/s]

16:53:23 | INFO | MRI (images): found 90 files


...

16:53:24 | INFO | MRI (masks): found 90 files


MRI/train/masks:   0%|          | 0/62 [00:00<?, ?it/s]

MRI/val/masks:   0%|          | 0/13 [00:00<?, ?it/s]

MRI/test/masks:   0%|          | 0/15 [00:00<?, ?it/s]

16:53:24 | INFO | Done — 534 slices in manifest, 0 volume issues
16:53:24 | INFO | Manifest saved → dataset_manifest.csv
16:53:24 | INFO | QC report  saved → qc_report.csv

--- Manifest preview (first 5 rows) ---
subject_id modality split  slice_index                                                                       image_path                                                                       mask_path                                                                       source_file
  11_CT_s4       CT  test            0 /Users/ayushchhetry/spartis assignment/dataset/test/CT/images/11_CT_s4_z0000.npy /Users/ayushchhetry/spartis assignment/dataset/test/CT/masks/11_CT_m4_z0000.npy /Users/ayushchhetry/spartis assignment/JUH_MR-CT_dataset/CT/image_CT/11_CT_s4.dcm
  11_CT_s4       CT  test            1 /Users/ayushchhetry/spartis assignment/dataset/test/CT/images/11_CT_s4_z0001.npy /Users/ayushchhetry/spartis assignment/dataset/test/CT/masks/11_CT_m4_z0001.npy /Users/ayushchhetry/spa